# 09 v2 · Artefacto en HF Hub + paquete de despliegue LOCAL de la API — Grupo 17 · Sprint 4, paso 1
**Cambio de arquitectura (D12):** HF ya no ofrece Spaces Docker gratuitos en CPU (respuesta 402 de la API de HF, 03/10/2026).
La API corre como **contenedor en el docker-compose local** (red del SOC); el artefacto sigue versionado en HF Hub (privado, tag v1).

**Entorno CPU.** Secrets en Colab: `HF_TOKEN_WRITE`, `HF_TOKEN_READ` y, opcionalmente, `API_KEY_G17`.

**Criterios de cierre:** (1) repo de modelo con tag `v1`; (2) la API, **cargando el modelo desde el Hub con el token de lectura**,
responde `/salud` = ok y 3 análisis válidos contra el esquema; (3) sin clave = 401; (4) carpeta `PI_G17/deploy/` lista para `docker compose`.

In [9]:
# Celda 1: Entorno y secretos
import os, sys, json, time, shutil, secrets, statistics, platform
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
%pip install -q "huggingface_hub>=0.24" jsonschema requests
from huggingface_hub import HfApi
TOKEN_W = userdata.get("HF_TOKEN_WRITE")
try: TOKEN_R = userdata.get("HF_TOKEN_READ")
except Exception: TOKEN_R = None
if not TOKEN_R:
    print("AVISO: sin HF_TOKEN_READ; el Space usará el token de escritura (menos seguro)."); TOKEN_R = TOKEN_W
try: API_KEY = userdata.get("API_KEY_G17"); NUEVA = False
except Exception: API_KEY, NUEVA = None, False
if not API_KEY: API_KEY, NUEVA = secrets.token_urlsafe(32), True
api = HfApi(token=TOKEN_W); USUARIO = api.whoami()["name"]
MODEL_REPO = f"{USUARIO}/g17-phishing-hibrido"
for f in ("api/app.py", "src/inference.py", "contracts/contrato_v1.schema.json", "models/fusion_v1.joblib", "models/distilbert_v1"):
    assert (BASE / f).exists(), f"Falta {f} en PI_G17"
print("usuario:", USUARIO, "| modelo:", MODEL_REPO)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
usuario: dennysSalazar | modelo: dennysSalazar/g17-phishing-hibrido


In [10]:
# Celda 2: Artefacto en el Hub (se omite si el tag v1 ya existe)
from huggingface_hub.utils import RepositoryNotFoundError
try:
    refs = api.list_repo_refs(MODEL_REPO, repo_type="model")
    YA_V1 = any(t.name == "v1" for t in refs.tags)
except RepositoryNotFoundError:
    YA_V1 = False
if YA_V1:
    print("El repo ya tiene el tag v1: no se vuelve a subir (inmutable).")
else:
    api.create_repo(MODEL_REPO, repo_type="model", private=True, exist_ok=True)
    ARTEFACTOS = ["distilbert_v1/*", "baseline_tfidf_lr_v1.joblib", "url_xgb_v1b_correos.json", "url_xgb_v1.meta.json",
                  "fusion_v1.joblib", "fusion_v1.meta.json"]
    api.upload_folder(folder_path=str(BASE / "models"), path_in_repo="models", repo_id=MODEL_REPO, repo_type="model",
                      allow_patterns=ARTEFACTOS, commit_message="Artefactos v1 (fase de modelado cerrada)")
    fus = json.loads((BASE / "results/fusion_v1.json").read_text()) if (BASE / "results/fusion_v1.json").exists() else {}
    tarjeta = f"""---
    license: other
    tags: [phishing, email-security, distilbert, xgboost, uees]
    ---
    # G17 · Detector híbrido de phishing (v1)
    Proyecto Integrador MIAR0545, UEES — Grupo 17. Clasifica correos en legítimo / spam / phishing con un riesgo calibrado 0–100.
    Fusión V3 (DistilBERT + TF-IDF/RL + XGBoost sobre URLs del correo), calibración isotónica.
    **Uso académico.** Datos de entrenamiento: CEAS_08, SpamAssassin, Nigerian_Fraud (Kaggle, Champa et al.), Nazario (CC BY 4.0),
    phishing_pot (CC BY-NC 4.0). Limitaciones: solo inglés; ~36 % de ruido de etiquetas en phishing_pot; ver informe técnico.
    Métricas (test, misma distribución): {json.dumps({k: fus.get('principal', {}).get('V3', {}).get('test_umbral', {}).get(k) for k in ('f1_macro', 'fpr_phishing')})}
    """
    api.upload_file(path_or_fileobj=tarjeta.encode(), path_in_repo="README.md", repo_id=MODEL_REPO, repo_type="model")
    try: api.create_tag(MODEL_REPO, tag="v1", repo_type="model")
    except Exception as e: print("tag v1:", type(e).__name__, "(si ya existía, se conserva)")
    print("artefactos publicados")
archivos = sorted(f for f in api.list_repo_files(MODEL_REPO, repo_type="model", revision="v1") if f.startswith("models/"))
print(len(archivos), "archivos en v1:", archivos[:8], "...")

El repo ya tiene el tag v1: no se vuelve a subir (inmutable).
9 archivos en v1: ['models/baseline_tfidf_lr_v1.joblib', 'models/distilbert_v1/config.json', 'models/distilbert_v1/model.safetensors', 'models/distilbert_v1/tokenizer.json', 'models/distilbert_v1/tokenizer_config.json', 'models/fusion_v1.joblib', 'models/fusion_v1.meta.json', 'models/url_xgb_v1.meta.json'] ...


In [11]:
# Celda 3: Carpeta del Space con versiones fijadas de ESTE entorno
import sklearn, xgboost, transformers, torch, numpy, scipy, pandas, joblib, jsonschema, huggingface_hub
SP = BASE / "deploy" / "api"; shutil.rmtree(SP, ignore_errors=True); SP.mkdir()
shutil.copy(BASE / "api/app.py", SP / "app.py")
for m in ("ioc_utils", "data_prep", "url_features", "inference"):
    shutil.copy(BASE / f"src/{m}.py", SP / f"{m}.py")
shutil.copy(BASE / "contracts/contrato_v1.schema.json", SP / "contrato_v1.schema.json")
VERS = {"python": platform.python_version(), "torch": torch.__version__.split("+")[0], "transformers": transformers.__version__,
        "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__, "numpy": numpy.__version__, "scipy": scipy.__version__,
        "pandas": pandas.__version__, "joblib": joblib.__version__}
(SP / "requirements.txt").write_text("\n".join([
    "--extra-index-url https://download.pytorch.org/whl/cpu", f"torch=={VERS['torch']}",
    *[f"{k}=={VERS[k]}" for k in ("transformers", "scikit-learn", "xgboost", "numpy", "scipy", "pandas", "joblib")],
    f"jsonschema=={jsonschema.__version__}", f"huggingface_hub=={huggingface_hub.__version__}",
    "fastapi>=0.110", "uvicorn[standard]>=0.29", "pydantic>=2"]) + "\n")
py = ".".join(VERS["python"].split(".")[:2])
(SP / "Dockerfile").write_text(f"""FROM python:{py}-slim
RUN useradd -m -u 1000 user
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=user . .
USER user
RUN mkdir -p /home/user/.cache/huggingface && chown -R user:user /home/user
ENV HOME=/home/user PYTHONUNBUFFERED=1 HF_HOME=/home/user/.cache/huggingface
EXPOSE 7860
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]
""")
# docker-compose (paso 1: solo la API; n8n, DFIR-IRIS y Wazuh se agregan en el paso 2)
DEP = BASE / "deploy"
(DEP / "docker-compose.yml").write_text("""name: g17-soc
services:
  api:
    build: ./api
    image: g17/phishing-api:v1
    env_file: .env
    volumes:
      - hf_cache:/home/user/.cache/huggingface      # el modelo se descarga una sola vez
    ports:
      - "127.0.0.1:8000:7860"                       # solo localhost (pruebas); n8n la llamará por la red interna
    networks: [soc, salida]
    mem_limit: 3g
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request,json,sys; sys.exit(0 if json.load(urllib.request.urlopen('http://localhost:7860/salud'))['estado']=='ok' else 1)"]
      interval: 30s
      timeout: 10s
      retries: 10
      start_period: 180s
networks:
  soc:
    internal: true        # red del SOC sin salida a internet
  salida: {}              # solo la API la usa, para descargar el modelo del Hub
volumes:
  hf_cache: {}
""")
(DEP / ".env.example").write_text("API_KEY=<clave de la API>\nHF_TOKEN=<token de LECTURA de Hugging Face>\n"
                                  f"MODEL_REPO={MODEL_REPO}\nMODEL_REVISION=v1\nDOMINIOS_PROPIOS=\n")
(DEP / ".gitignore").write_text(".env\n")
(DEP / "README.md").write_text("# Despliegue local — Grupo 17\n1. Copiar `.env.example` a `.env` y completar API_KEY y HF_TOKEN (lectura).\n"
                               "2. `docker compose up -d --build api`\n3. `docker compose ps` hasta ver `healthy`.\n"
                               "4. Prueba: ver la Celda 5 del notebook 09 v2.\n")
print(VERS); print("deploy/:", sorted(str(p.relative_to(BASE / "deploy")) for p in (BASE / "deploy").rglob("*") if p.is_file()))

/tmp/ipykernel_4253/550359797.py:14: DeprecationWarning: Accessing jsonschema.__version__ is deprecated and will be removed in a future release. Use importlib.metadata directly to query for jsonschema's version.
  f"jsonschema=={jsonschema.__version__}", f"huggingface_hub=={huggingface_hub.__version__}",


{'python': '3.13.15', 'torch': '2.11.0', 'transformers': '5.17.0', 'scikit-learn': '1.6.1', 'xgboost': '3.4.1', 'numpy': '2.1.3', 'scipy': '1.16.3', 'pandas': '2.2.3', 'joblib': '1.6.0'}
deploy/: ['.env.example', '.gitignore', 'README.md', 'api/Dockerfile', 'api/app.py', 'api/contrato_v1.schema.json', 'api/data_prep.py', 'api/inference.py', 'api/ioc_utils.py', 'api/requirements.txt', 'api/url_features.py', 'docker-compose.yml']


In [12]:
# Celda 4: La API real en Colab, cargando el modelo DESDE EL HUB
# Prueba la cadena completa de la capa 3: snapshot_download(v1, token de lectura) -> Detector -> FastAPI -> contrato.
%pip install -q fastapi httpx "uvicorn[standard]"
import importlib, requests, jsonschema
os.environ.update({"MODEL_REPO": MODEL_REPO, "MODEL_REVISION": "v1", "HF_TOKEN": TOKEN_R, "API_KEY": API_KEY})
sys.path.insert(0, str(BASE / "deploy" / "api"))
import app as A; importlib.reload(A)
from fastapi.testclient import TestClient
ESQ = jsonschema.Draft202012Validator(json.loads((BASE / "contracts/contrato_v1.schema.json").read_text()), format_checker=jsonschema.FormatChecker())
CORREOS = [("Account suspended", "Dear user, verify your password at https://secure-login.example.com/verify?id=7 today."),
           ("Reunión", "Hola equipo, nos vemos el martes para revisar el informe."),
           ("Big sale", "Replica watches 80% off, visit www.shop.example.net/promo now")]
t0 = time.time()
with TestClient(A.crear_app()) as c:
    ARRANQUE_S = round(time.time() - t0, 1); salud = c.get("/salud").json()
    validas, clases, lat = 0, [], []
    for a, cu in CORREOS * 7:
        t1 = time.time(); j = c.post("/v1/analizar", headers={"X-API-Key": API_KEY}, json={"asunto": a, "cuerpo": cu}).json()
        lat.append(1000 * (time.time() - t1))
        try: ESQ.validate(j); validas += 1
        except Exception: pass
        clases.append(j.get("clasificacion", {}).get("clase"))
    sin_clave = c.post("/v1/analizar", json={"cuerpo": "x"}).status_code
PRUEBA = {"salud": salud, "arranque_con_descarga_s": ARRANQUE_S, "llamadas": len(lat), "validas_contra_esquema": validas,
          "clases_primeras_3": clases[:3], "latencia_ms": {"media": round(statistics.mean(lat), 1), "p95": round(sorted(lat)[int(.95 * len(lat)) - 1], 1)},
          "sin_clave_http": sin_clave}
if NUEVA:
    # La clave generada se mostraba aquí para copiarla a deploy/.env. En la versión publicada
    # del notebook la salida se retiró: una clave nunca debe quedar en un archivo versionado.
    print(">>> API_KEY generada: copiarla al secret API_KEY_G17 y a deploy/.env")
PRUEBA

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

>>> API_KEY generada: copiarla al secret API_KEY_G17 y a deploy/.env
[valor retirado de la versión publicada]


{'salud': {'estado': 'ok',
  'version_api': '1.0',
  'version_contrato': '1.0',
  'modelo': 'fusion_v1',
  'variante': 'V3',
  'carga_s': 30.8,
  'error_carga': None},
 'arranque_con_descarga_s': 31.0,
 'llamadas': 21,
 'validas_contra_esquema': 21,
 'clases_primeras_3': ['phishing', 'phishing', 'spam'],
 'latencia_ms': {'media': 89.8, 'p95': 90.1},
 'sin_clave_http': 401}

In [13]:
# Celda 5: RESUMEN (sin secretos)
RESUMEN = {"model_repo": MODEL_REPO, "tag_v1_ya_existia": YA_V1, "versiones_fijadas": VERS, "prueba_api_desde_hub": PRUEBA,
           "aceptacion": {"salud_ok": PRUEBA["salud"].get("estado") == "ok", "contrato_100pct": PRUEBA["validas_contra_esquema"] == PRUEBA["llamadas"],
                          "auth_401": PRUEBA["sin_clave_http"] == 401, "deploy_listo": (BASE / "deploy/docker-compose.yml").exists()}}
(BASE / "results/despliegue_v1.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (09) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (09) =====
{"model_repo":"dennysSalazar/g17-phishing-hibrido","tag_v1_ya_existia":true,"versiones_fijadas":{"python":"3.13.15","torch":"2.11.0","transformers":"5.17.0","scikit-learn":"1.6.1","xgboost":"3.4.1","numpy":"2.1.3","scipy":"1.16.3","pandas":"2.2.3","joblib":"1.6.0"},"prueba_api_desde_hub":{"salud":{"estado":"ok","version_api":"1.0","version_contrato":"1.0","modelo":"fusion_v1","variante":"V3","carga_s":30.8,"error_carga":null},"arranque_con_descarga_s":31.0,"llamadas":21,"validas_contra_esquema":21,"clases_primeras_3":["phishing","phishing","spam"],"latencia_ms":{"media":89.8,"p95":90.1},"sin_clave_http":401},"aceptacion":{"salud_ok":true,"contrato_100pct":true,"auth_401":true,"deploy_listo":true}}
